<div style="text-align: center;">
  <span style="color: lightskyblue; font-weight: bold; font-size: 50px;">
    Demo File for Pick and Place
  </span>
</div>

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Import Library
</span>

In [1]:
import rby1_sdk as rby
import numpy as np
import rby1_lib_ysw as ysw
import logging
import sys
import time
import pyrealsense2 as rs
import cv2

np.set_printoptions(suppress=True, precision=5, linewidth=300)  

# In .ipynb, show logging 
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s",
    # format="%(asctime)s [%(levelname)s] %(message)s",
    stream=sys.stdout,
    force=True,
)

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Initialization [Connect + PowerON]
</span>

In [2]:
# robotConfig = ysw.setRobotConfig(ysw.MODE_SIM, ysw.MODEL_A)                 # For Simulation
robotConfig = ysw.setRobotConfig(ysw.MODE_LOCAL_COMPUTER, ysw.MODEL_A)      # For Real Robot + Laptop (RBY1-Hotspot)
# robotConfig = ysw.setRobotConfig(ysw.MODE_DIRECT_UPC, ysw.MODEL_A)          # For Real Robot + UPC

robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_AND_POWER)
# robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_ONLY)
if (robot != None):
    model = robot.model()

*************************** [Initialization Starts] ****************************

[INFO] SUCCESS: Connected to the Robot!
[INFO] SUCCESS: Power ON the Robot!
[INFO] SUCCESS: Servo ON the Robot!
[INFO] SUCCESS: Reset the Control Manager!
[INFO] SUCCESS: Enable the Control Manager!

*************************** [Intialization is Done] ****************************





<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Power OFF
</span>

In [ ]:
ysw.powerOFF(robot, robotConfig)

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Posture Change
</span>

In [3]:
Desired_Pos = ysw.DEMO_POS     # ZERO_POS, READY_POS, PACK_POS, DEMO_POS
ysw.positionCtrl_joint_22dof(robot, Desired_Pos, minimum_time=5.0, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="color:gray; font-weight:bold; font-size:50px;">
Get Forward Kinematics
</span>

In [4]:
# [Link]:     "link_right_arm_6" 
# -> [Joint]: "tool_right"
# -> [Link]:  "ee_right" 
# -> [Joint]: "gripper_finger_r1"
# -> [Link]:  "ee_finger_r1"    (r1이 ee_right 기준으로 +x 방향, 즉 정면에서 볼때 가까운 방향의 그리퍼임)

# ysw.get_FK(robot, reference_link="base", target_link="link_torso_5")
# ysw.get_FK(robot, reference_link="base", target_link="ee_right")
# ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_right")
ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_finger_r1")

*************** [FK of 'ee_finger_r1' from 'link_head_2' Starts] ***************

[Result]

T:
[[ 0.19674  0.17101 -0.96543  0.54588]
 [ 0.262    0.93969  0.21984 -0.3291 ]
 [ 0.9448  -0.2962   0.14007 -0.10022]
 [ 0.       0.       0.       1.     ]]


position [m]: [ 0.54588 -0.3291  -0.10022]
rpy [rad]: [-1.12906 -1.23698  0.92671]
rpy [deg]: [-64.69032 -70.87389  53.09641]

**************** [FK of 'ee_finger_r1' from 'link_head_2' Ends] ****************





{'T': array([[ 0.19674,  0.17101, -0.96543,  0.54588],
        [ 0.262  ,  0.93969,  0.21984, -0.3291 ],
        [ 0.9448 , -0.2962 ,  0.14007, -0.10022],
        [ 0.     ,  0.     ,  0.     ,  1.     ]]),
 'position': array([ 0.54588, -0.3291 , -0.10022]),
 'rotation': array([[ 0.19674,  0.17101, -0.96543],
        [ 0.262  ,  0.93969,  0.21984],
        [ 0.9448 , -0.2962 ,  0.14007]]),
 'rpy': array([-1.12906, -1.23698,  0.92671])}

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Position Control based on D435i frame
</span>

In [18]:
### Move to Pre-posture (Joint Position Control)
pre_posture = ysw.DEMO_POS     # ZERO_POS, READY_POS, PACK_POS, DEMO_POS
ysw.positionCtrl_joint_22dof(robot, pre_posture, minimum_time=5.0, control_hold_time=0.5)


### FK example
# Execute: ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_finger_r1")
# Result: T:
            # [[ 0.19674  0.17101 -0.96543  0.54588]
            # [ 0.262    0.93969  0.21984 -0.3291 ]
            # [ 0.9448  -0.2962   0.14007 -0.10022]
            # [ 0.       0.       0.       1.     ]]
            
            # position [m]: [ 0.54588 -0.3291  -0.10022]
            # rpy [rad]: [-1.12906 -1.23698  0.92671]
            # rpy [deg]: [-64.69032 -70.87389  53.09641]
            
fk_result = ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_finger_r1")            
pos = fk_result["position"]
rpy = fk_result["rpy"]
            
R1 = ysw.rotMat_y(-90.0*ysw.DEG2RAD)    # "link_head_2" frame 기준으로 y축 -90도 회전
R2 = ysw.rotMat_x(90.0*ysw.DEG2RAD)     # "link_head_2" frame 기준으로 x축 +90도 회전
R = R1 @ R2
rpy_changed = ysw.rotMat_to_rpy(R)


### Cartesian Position Control example (Whole-Body version)
cartesian_position_targets_sample = [    
                                {
                                    "reference_link": "link_head_2",
                                    "target_link": "ee_finger_r1",
                                    "pos": pos + [0.1, 0.0, 0],                     # 현재 위치는 유지
                                    "rpy": rpy,                     # 자세만 link_head_2 기준으로 변경
                                },                 
                            ]


joint_targets_sample = {
                        "right_arm_6": 0.0 * ysw.DEG2RAD,
                        "left_arm_6": 0.0 * ysw.DEG2RAD,
                       }

# [rv = "function"] form is Necessary when "get_return=False"
rv = ysw.positionCtrl_cartesian_wholeBody(
    robot, 
    position_targets=cartesian_position_targets_sample, 
    # joint_targets=joint_targets_sample, 
    joint_targets=None,
    get_return=True
)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************



*************** [FK of 'ee_finger_r1' from 'link_head_2' Starts] ***************

[Result]

T:
[[ 0.19679  0.17094 -0.96543  0.55015]
 [ 0.26219  0.93965  0.21982 -0.32316]
 [ 0.94474 -0.29638  0.1401  -0.14841]
 [ 0.       0.       0.       1.     ]]


position [m]: [ 0.55015 -0.32316 -0.14841]
rpy [rad]: [-1.12922 -1.2368   0.92692]
rpy [deg]: [-64.69976 -70.8632   53.10834]

**************** [FK of 'ee_finger_r1' from 'link_head_2' Ends] ****************



*************** [Cartesian Position Control(Whole-Body) Starts] ****************

[INFO] Add (Whole-Body) Cartesian target 0: 'link_head_2'->'ee_finger_r1', pos=[ 0.65015 -0.32316 -0.14841], rpy=[-1.12922 -1.2368   0.92692]

**************** [Cartesian Position Control(Whole-Body) Ends] *****************





In [4]:
d435i = ysw.D435i(rgb_on=True, depth_on=True, pixel=(640,480), fps=30)
d435i.initialize()

d435i.startPipeline()

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)

        detected_objects, color_image, mask = d435i.detectColorObjectContour(
            color_image=color_image,
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_BLACK,
            minimum_area=300                    # Can be changed
        )

        for obj in detected_objects:
            print(
                f"[object {obj['id']}] "
                f"pixel={obj['pixel']}, "
                f"pos={np.round(obj['pos'], 3)}, "
                f"area={obj['area']:.1f}"
            )

        d435i.imshow("color_image", color_image)
        d435i.imshow("depth_image", depth_colorMap)
        d435i.imshow("mask", mask)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()

[ERROR] Failed to start D435i pipeline: No device connected
[ERROR] Frame didn't arrive: wait_for_frames cannot be called before start()
[ERROR] Frame didn't arrive: wait_for_frames cannot be called before start()
[ERROR] Frame didn't arrive: wait_for_frames cannot be called before start()
[ERROR] Frame didn't arrive: wait_for_frames cannot be called before start()
[ERROR] Frame didn't arrive: wait_for_frames cannot be called before start()
[ERROR] Frame didn't arrive: wait_for_frames cannot be called before start()
[ERROR] Frame didn't arrive: wait_for_frames cannot be called before start()
[ERROR] Frame didn't arrive: wait_for_frames cannot be called before start()
[ERROR] Frame didn't arrive: wait_for_frames cannot be called before start()
[ERROR] Frame didn't arrive: wait_for_frames cannot be called before start()
[ERROR] Frame didn't arrive: wait_for_frames cannot be called before start()
[ERROR] Frame didn't arrive: wait_for_frames cannot be called before start()
[ERROR] Frame di

KeyboardInterrupt: 